In [17]:
from datasets import load_dataset
import pandas as pd

In [18]:
DATASET_NAME = "fancyzhx/amazon_polarity"

TARGET_PER_CLASS = 10_000
RANDOM_SEED = 42

OUTPUT_FILE = r"E:\Rasengan\product-review\data\amazon_reviews_20000.csv"

In [19]:
dataset = load_dataset(
    DATASET_NAME,
    split="train",
    streaming=True
)

dataset = dataset.shuffle(
    seed=RANDOM_SEED,
    buffer_size=100_000
)

In [20]:
reviews = {
    0: [],
    1: []
}

scanned = 0

for item in dataset:
    scanned += 1

    label = item["label"]

    if len(reviews[label]) >= TARGET_PER_CLASS:
        continue

    title = (item["title"] or "").strip()
    content = (item["content"] or "").strip()

    review_text = " ".join(
        part for part in [title, content] if part
    )

    if not review_text:
        continue

    reviews[label].append({
        "review_text": review_text,
        "label": label
    })

    if all(
        len(reviews[c]) >= TARGET_PER_CLASS
        for c in [0, 1]
    ):
        break

print("Reviews scanned:", scanned)
print("Negative reviews:", len(reviews[0]))
print("Positive reviews:", len(reviews[1]))

Reviews scanned: 20070
Negative reviews: 10000
Positive reviews: 10000


In [21]:
df = pd.DataFrame(
    reviews[0] + reviews[1]
)

df["sentiment"] = df["label"].map({
    0: "BAD",
    1: "GOOD"
})

df = df.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (20000, 3)


,review_text,label,sentiment
0,"Good diapers, some blowouts I like these diapers and it's great to get so many at once, but we have had some blowouts up the back.",1,GOOD
1,"Broke within a month Even though I always peeled the cloves, this press didn't do a good job fully pressing the garlic. And then, after only a few weeks, it snapped in two as I was pressing a clov...",0,BAD
2,"Manipulative, offensive, innacurate As others said, the author really botched it with the main character: to imagine that any able-minded German 9-year-old, and a son of a high ranking SS commande...",0,BAD
3,The Truman Show for Dummies A watered-down Truman Show for people who couldn't understand or don't want to deal with the larger concepts in The Truman Show.,0,BAD
4,"Straight Fire! Holy Culture was my introduction to Cross Movement and since hearing it, I have purchased every CD they released including the solo projects and their old albulms. I have to be hone...",1,GOOD


In [22]:
df["sentiment"].value_counts()

sentiment
GOOD    10000
BAD     10000
Name: count, dtype: int64

In [23]:
df.isnull().sum()

review_text    0
label          0
sentiment      0
dtype: int64

In [24]:
print(
    "Exact duplicate reviews:",
    df["review_text"].duplicated().sum()
)

Exact duplicate reviews: 0


In [25]:
pd.set_option("display.max_colwidth", 200)

df.sample(
    10,
    random_state=RANDOM_SEED
)[["review_text", "label", "sentiment"]]

,review_text,label,sentiment
10650,"Disappointing I bought this book because I am a nonogram (also called Paint By Number) addict, and can't get enough of this type of puzzle. My disappointment is not so much with the quality of the...",0,BAD
2041,"Here's the thing. This Sunflower/ Surf's Up set is really good and everything, but what you NEED to know is this: If you pass over this album, do yourself a favor and download A Day in the Life of...",1,GOOD
8668,Ineffective for promoting vivid dreams I bought this product because I heard from my Native American friends that it promotes vivid dreams. I have used it under my pillow for three weeks and no lu...,0,BAD
1114,"It's not just about a sheep, people! This book takes thought. It takes patience. It takes an interest in the nitty gritty of politics and history in the making. It is probably an oversimplication ...",1,GOOD
13902,"Excellent helper for new parents This book went into just the right amount of detail for me. It had plenty of detailed information on what to expect in terms of sleeping, eating, etc., at differen...",1,GOOD
11963,"Dark Dark Tale The artwork is beautiful and detailed. My three year old loves to find the cat in each picture and to find the faces in the windows, walls, and curtains. One can always find somethi...",1,GOOD
11072,"Awesome Calculator Watch For those of you in the market for a calculator watch this is worth every penny, all 1492 of them. (Came in handy again right there...) The main thing is that it's light, ...",1,GOOD
3002,"too depressing, not realistic I have read most of John Grisham's earlier books, and enjoyed them, so I thought I would try this one. To me, it was not quite as good as his others. I thought this b...",0,BAD
19771,"Crap Look, I don't get what the hype about this album is. Nevermore can obviously play, as demonstrated by the first song on here. But why did they choose to have the whole album be a mid-paced, b...",0,BAD
8115,"Disappointing A vividly imagined tale of what might have been had the infamous Lucky Lucan (wanted for murder) come to Paris with his body double, settled in for overpriced psychotherapy with a fo...",0,BAD


In [26]:
df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8"
)

print(f"Saved dataset to: {OUTPUT_FILE}")

Saved dataset to: E:\Rasengan\product-review\data\amazon_reviews_20000.csv


In [27]:
df_check = pd.read_csv(OUTPUT_FILE)

print("Saved shape:", df_check.shape)
print("\nClass distribution:")
print(df_check["sentiment"].value_counts())

df_check.head()

Saved shape: (20000, 3)

Class distribution:
sentiment
GOOD    10000
BAD     10000
Name: count, dtype: int64


,review_text,label,sentiment
0,"Good diapers, some blowouts I like these diapers and it's great to get so many at once, but we have had some blowouts up the back.",1,GOOD
1,"Broke within a month Even though I always peeled the cloves, this press didn't do a good job fully pressing the garlic. And then, after only a few weeks, it snapped in two as I was pressing a clov...",0,BAD
2,"Manipulative, offensive, innacurate As others said, the author really botched it with the main character: to imagine that any able-minded German 9-year-old, and a son of a high ranking SS commande...",0,BAD
3,The Truman Show for Dummies A watered-down Truman Show for people who couldn't understand or don't want to deal with the larger concepts in The Truman Show.,0,BAD
4,"Straight Fire! Holy Culture was my introduction to Cross Movement and since hearing it, I have purchased every CD they released including the solo projects and their old albulms. I have to be hone...",1,GOOD
